# Notebook 08 -- Feature Extraction: Text

<div style="border-left: 4px solid #4680a7; padding: 10px 15px; margin: 10px 0; background: #4e6681;">

**Objective:** Extract semantic and statistical features from the `Description` column and pre-computed Google NLP sentiment JSONs. Produce dense embedding representations using a pretrained sentence transformer (`all-mpnet-base-v2`), complemented by handcrafted text statistics and sentiment aggregates.

**Answers:** *"What textual signals -- semantic, lexical, and sentiment -- can be captured as features for adoption speed prediction?"*

</div>

| Item | Detail |
|------|--------|
| **Dependencies** | `data/cleaned/train.parquet`, `data/cleaned/test.parquet`, `data/raw/train_sentiment/`, `data/raw/test_sentiment/` |
| **Artifacts** | `data/features/text/v1/train.parquet`, `data/features/text/v1/test.parquet`, `data/features/text/v1/schema.json`, `reports/feature_extraction_text.json` |
| **Model** | `sentence-transformers/all-mpnet-base-v2` (768-dim embeddings, max 384 tokens) |
| **Scope** | Text and sentiment features only -- no tabular, image, or metadata features |
| **Runtime** | ~5-15 min (depends on GPU/CPU availability) |

---
## 1. Imports & Configuration

In [1]:
# -- Standard library & third-party imports ------------------------------------
from __future__ import annotations

import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# -- Project imports -----------------------------------------------------------
from adoption_accelerator import config as cfg
from adoption_accelerator.data.ingestion import load_parquet, save_parquet
from adoption_accelerator.features.text import (
    FEATURE_DESCRIPTIONS,
    SENTIMENT_COLUMNS,
    TEXT_STAT_COLUMNS,
    compute_text_statistics,
    detect_languages,
    extract_embeddings,
    extract_text_features,
    preprocess_descriptions,
    reduce_embedding_dimensions,
)
from adoption_accelerator.features.metadata import aggregate_sentiment_features
from adoption_accelerator.features.registry import (
    build_column_descriptors,
    compute_config_hash,
    save_feature_schema,
)
from adoption_accelerator.utils.logging import setup_logging

setup_logging()

# -- Reproducibility & display ------------------------------------------------
SEED = cfg.SEED
np.random.seed(SEED)
random.seed(SEED)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:.4f}".format)

# -- Text feature configuration ------------------------------------------------
TEXT_CONFIG = {
    "model_name": "sentence-transformers/all-mpnet-base-v2",
    "embedding_dim": 768,
    "max_seq_length": 384,
    "batch_size": 64,
    "device": None,       # auto-detect (cuda if available, else cpu)
    "apply_pca": False,   # disabled unless explicitly needed
    "pca_components": 100,
}

# -- Output paths --------------------------------------------------------------
FEATURES_DIR = cfg.DATA_FEATURES / "text" / "v1"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = cfg.REPORTS_DIR / "feature_extraction_text.json"

print("Imports and configuration loaded.")
print(f"   Seed          : {SEED}")
print(f"   Model         : {TEXT_CONFIG['model_name']}")
print(f"   Embedding dim : {TEXT_CONFIG['embedding_dim']}")
print(f"   Batch size    : {TEXT_CONFIG['batch_size']}")
print(f"   PCA enabled   : {TEXT_CONFIG['apply_pca']}")
print(f"   Features dir  : {FEATURES_DIR}")
print(f"   Report path   : {REPORT_PATH}")

Imports and configuration loaded.
   Seed          : 42
   Model         : sentence-transformers/all-mpnet-base-v2
   Embedding dim : 768
   Batch size    : 64
   PCA enabled   : False
   Features dir  : C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\text\v1
   Report path   : C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\reports\feature_extraction_text.json


---
## 2. Load Cleaned Data

In [2]:
# -- Load cleaned train and test DataFrames ------------------------------------
train = load_parquet(cfg.DATA_CLEANED / "train.parquet")
test = load_parquet(cfg.DATA_CLEANED / "test.parquet")

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"\nTrain columns with text:")
print(f"   PetID       : {train['PetID'].nunique():,} unique")
print(f"   Description : {train['Description'].notna().sum():,} non-null / {train['Description'].isna().sum():,} null")

09:50:46  INFO      Loaded Parquet: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\cleaned\train.parquet (14993 rows, 25 cols)
09:50:46  INFO      Loaded Parquet: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\cleaned\test.parquet (3972 rows, 24 cols)


Train shape : (14993, 25)
Test shape  : (3972, 24)

Train columns with text:
   PetID       : 14,993 unique
   Description : 14,980 non-null / 13 null


In [3]:
# -- Validation Gate G08-1: Required columns present ---------------------------
EXPECTED_TRAIN_ROWS = 14_993
EXPECTED_TEST_ROWS = 3_972

assert "PetID" in train.columns and "PetID" in test.columns, "G08-1 FAIL: PetID missing"
assert "Description" in train.columns and "Description" in test.columns, "G08-1 FAIL: Description missing"
assert train.shape[0] == EXPECTED_TRAIN_ROWS, (
    f"G08-1 FAIL: Expected {EXPECTED_TRAIN_ROWS} train rows, got {train.shape[0]}"
)
assert test.shape[0] == EXPECTED_TEST_ROWS, (
    f"G08-1 FAIL: Expected {EXPECTED_TEST_ROWS} test rows, got {test.shape[0]}"
)

print("G08-1 PASS: Cleaned data loaded with PetID and Description columns.")
print(f"   Train: {train.shape[0]:,} rows x {train.shape[1]} cols")
print(f"   Test : {test.shape[0]:,} rows x {test.shape[1]} cols")

G08-1 PASS: Cleaned data loaded with PetID and Description columns.
   Train: 14,993 rows x 25 cols
   Test : 3,972 rows x 24 cols


---
## 3. Text Preprocessing

<div style="border-left: 4px solid #f39c12; padding: 8px 12px; margin: 8px 0; background: #5e5535;">

**Strategy:** Replace null/empty descriptions with a canonical placeholder, strip HTML and URLs, normalize whitespace and Unicode (NFKD). Casing is preserved (transformer tokenizers handle casing natively). No stop-word removal.

</div>

In [4]:
# -- Preprocess train descriptions ---------------------------------------------
train_desc_clean, train_preprocess_stats = preprocess_descriptions(train["Description"])

print("Train preprocessing statistics:")
for k, v in train_preprocess_stats.items():
    print(f"   {k:<25s}: {v}")

print(f"\nSample preprocessed descriptions:")
for i, (pid, desc) in enumerate(zip(train["PetID"].head(3), train_desc_clean.head(3))):
    snippet = desc[:120] + "..." if len(desc) > 120 else desc
    print(f"   [{pid}] {snippet}")

09:51:34  INFO      Text preprocessing complete: 14993 descriptions (13 null, 13 empty/null replaced)


Train preprocessing statistics:
   n_null                   : 13
   n_empty_or_null          : 13
   n_cleaned                : 14993
   char_length_mean         : 339.2
   char_length_median       : 238.0
   char_length_min          : 1
   char_length_max          : 6664

Sample preprocessed descriptions:
   [86e1089a3] Nibble is a 3+ month old ball of cuteness. He is energetic and playful. I rescued a couple of cats a few months ago but ...
   [6296e909a] I just found it alone yesterday near my apartment. It was shaking so I had to bring it home to provide temporary care.
   [3422e4906] Their pregnant mother was dumped by her irresponsible owner at the roadside near some shops in Subang Jaya. Gave birth t...


In [5]:
# -- Preprocess test descriptions ----------------------------------------------
test_desc_clean, test_preprocess_stats = preprocess_descriptions(test["Description"])

print("Test preprocessing statistics:")
for k, v in test_preprocess_stats.items():
    print(f"   {k:<25s}: {v}")

09:51:36  INFO      Text preprocessing complete: 3972 descriptions (1 null, 1 empty/null replaced)


Test preprocessing statistics:
   n_null                   : 1
   n_empty_or_null          : 1
   n_cleaned                : 3972
   char_length_mean         : 311.0
   char_length_median       : 232.0
   char_length_min          : 2
   char_length_max          : 3556


---
## 4. Language Detection

In [6]:
# -- Detect languages on a sample of train descriptions ------------------------
lang_report = detect_languages(train_desc_clean, sample_size=1_000, seed=SEED)

print("Language detection (sample n=%d):" % lang_report["sample_size"])
print(f"   Dominant language   : {lang_report['dominant_language']}")
print(f"   Multilingual ratio  : {lang_report['multilingual_ratio']:.2%}")
print(f"\n   Language distribution (top 10):")
sorted_langs = sorted(lang_report["language_counts"].items(), key=lambda x: -x[1])
for lang, count in sorted_langs[:10]:
    print(f"     {lang:<6s}: {count:>4d} ({count / lang_report['sample_size']:.1%})")

09:51:51  INFO      Language detection: dominant=en, multilingual_ratio=7.00% (n=1000)


Language detection (sample n=1000):
   Dominant language   : en
   Multilingual ratio  : 7.00%

   Language distribution (top 10):
     en    :  930 (93.0%)
     id    :   35 (3.5%)
     da    :   11 (1.1%)
     ro    :    6 (0.6%)
     zh-cn :    4 (0.4%)
     no    :    2 (0.2%)
     hr    :    2 (0.2%)
     de    :    2 (0.2%)
     es    :    2 (0.2%)
     af    :    2 (0.2%)


---
## 5. Handcrafted Text Statistics

In [7]:
# -- Compute text statistics for train -----------------------------------------
train_text_stats = compute_text_statistics(train_desc_clean)

print(f"Text statistics shape: {train_text_stats.shape}")
print(f"\nTrain text statistics summary:")
display(
    train_text_stats.describe()
    .T.style.format("{:.2f}")
    .set_caption("Train Description Text Statistics")
)

09:52:15  INFO      Computed text statistics: 14993 rows x 7 features


Text statistics shape: (14993, 7)

Train text statistics summary:


,count,mean,std,min,25%,50%,75%,max
description_length,14993.00,339.22,373.34,1.00,117.00,238.00,431.00,6664.00
word_count,14993.00,62.95,69.30,1.00,21.00,44.00,81.00,1257.00
sentence_count,14993.00,6.16,5.16,1.00,3.00,5.00,8.00,86.00
avg_word_length,14993.00,4.62,2.00,1.00,4.13,4.41,4.80,118.00
uppercase_ratio,14993.00,0.04,0.07,0.00,0.02,0.03,0.04,1.00
punctuation_ratio,14993.00,0.04,0.06,0.00,0.02,0.03,0.04,1.00
digit_ratio,14993.00,0.00,0.01,0.00,0.00,0.00,0.01,1.00


In [8]:
# -- Compute text statistics for test ------------------------------------------
test_text_stats = compute_text_statistics(test_desc_clean)

print(f"Test text statistics shape: {test_text_stats.shape}")
print(f"Null counts: {int(test_text_stats.isna().sum().sum())}")

09:52:15  INFO      Computed text statistics: 3972 rows x 7 features


Test text statistics shape: (3972, 7)
Null counts: 0


---
## 6. Embedding Extraction

<div style="border-left: 4px solid #f39c12; padding: 8px 12px; margin: 8px 0; background: #5e5535;">

**Model:** `all-mpnet-base-v2` -- MPNet-based sentence transformer producing 768-dimensional embeddings. Highest quality among sentence-transformers models on STS benchmarks. Max sequence length: 384 tokens.

</div>

In [9]:
# -- Extract embeddings for TRAIN descriptions ---------------------------------
import time

t0 = time.perf_counter()
train_embeddings = extract_embeddings(
    train_desc_clean,
    model_name=TEXT_CONFIG["model_name"],
    batch_size=TEXT_CONFIG["batch_size"],
    device=TEXT_CONFIG["device"],
)
train_emb_time = time.perf_counter() - t0

print(f"Train embeddings: {train_embeddings.shape}")
print(f"   dtype : {train_embeddings.dtype}")
print(f"   time  : {train_emb_time:.1f}s")
print(f"   memory: {train_embeddings.nbytes / 1_048_576:.1f} MB")

c:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
09:52:37  INFO      Loading sentence transformer model: sentence-transformers/all-mpnet-base-v2 (batch_size=64)
c:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\cliente\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For mo

Train embeddings: (14993, 768)
   dtype : float32
   time  : 497.5s
   memory: 43.9 MB


In [10]:
# -- Extract embeddings for TEST descriptions ----------------------------------
t0 = time.perf_counter()
test_embeddings = extract_embeddings(
    test_desc_clean,
    model_name=TEXT_CONFIG["model_name"],
    batch_size=TEXT_CONFIG["batch_size"],
    device=TEXT_CONFIG["device"],
)
test_emb_time = time.perf_counter() - t0

print(f"Test embeddings : {test_embeddings.shape}")
print(f"   dtype : {test_embeddings.dtype}")
print(f"   time  : {test_emb_time:.1f}s")
print(f"   memory: {test_embeddings.nbytes / 1_048_576:.1f} MB")

10:00:51  INFO      Loading sentence transformer model: sentence-transformers/all-mpnet-base-v2 (batch_size=64)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1160.37it/s, Materializing param=pooler.dense.weight]                       
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
10:00:54  INFO      Extracting embeddings for 3972 texts ...
Batches: 100%|██████████| 63/63 [01:52<00:00,  1.78s/it]
10:02:46  INFO      Embeddings extracted: shape=(3972, 768), dtype=float32


Test embeddings : (3972, 768)
   dtype : float32
   time  : 115.0s
   memory: 11.6 MB


In [11]:
# -- Embedding shape validation ------------------------------------------------
expected_dim = TEXT_CONFIG["embedding_dim"]

assert train_embeddings.shape == (EXPECTED_TRAIN_ROWS, expected_dim), (
    f"Train embedding shape mismatch: {train_embeddings.shape} != ({EXPECTED_TRAIN_ROWS}, {expected_dim})"
)
assert test_embeddings.shape == (EXPECTED_TEST_ROWS, expected_dim), (
    f"Test embedding shape mismatch: {test_embeddings.shape} != ({EXPECTED_TEST_ROWS}, {expected_dim})"
)

# Check for degenerate (all-zero) embeddings
train_zero_rows = int(np.all(train_embeddings == 0, axis=1).sum())
test_zero_rows = int(np.all(test_embeddings == 0, axis=1).sum())

print(f"Embedding dimension check: PASS ({expected_dim}-dim)")
print(f"   Train all-zero rows: {train_zero_rows}")
print(f"   Test all-zero rows : {test_zero_rows}")

if train_zero_rows > 0 or test_zero_rows > 0:
    print("   [WARN] Degenerate embeddings detected -- check empty descriptions.")

Embedding dimension check: PASS (768-dim)
   Train all-zero rows: 0
   Test all-zero rows : 0


---
## 7. Sentiment Feature Aggregation

In [12]:
# -- Aggregate sentiment features from JSON files ------------------------------
print("Loading and aggregating train sentiment JSONs...")
train_sentiment = aggregate_sentiment_features(split="train")

print(f"\nLoading and aggregating test sentiment JSONs...")
test_sentiment = aggregate_sentiment_features(split="test")

print(f"\nSentiment coverage:")
train_coverage = train_sentiment["PetID"].isin(train["PetID"]).sum() / len(train)
test_coverage = test_sentiment["PetID"].isin(test["PetID"]).sum() / len(test)
print(f"   Train: {train_coverage:.2%} ({len(train_sentiment):,} sentiment files / {len(train):,} pets)")
print(f"   Test : {test_coverage:.2%} ({len(test_sentiment):,} sentiment files / {len(test):,} pets)")

Loading and aggregating train sentiment JSONs...


10:03:10  INFO      Found 14442 sentiment JSON files in C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\raw\train_sentiment
10:03:48  INFO        ... loaded 5000 / 14442 sentiment JSONs
10:04:27  INFO        ... loaded 10000 / 14442 sentiment JSONs
10:05:03  INFO      Aggregated sentiment features for split='train': 14442 rows x 10 cols



Loading and aggregating test sentiment JSONs...


10:05:04  INFO      Found 3865 sentiment JSON files in C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\raw\test_sentiment
10:05:35  INFO      Aggregated sentiment features for split='test': 3865 rows x 10 cols



Sentiment coverage:
   Train: 96.32% (14,442 sentiment files / 14,993 pets)
   Test : 97.31% (3,865 sentiment files / 3,972 pets)


In [13]:
# -- Preview sentiment features ------------------------------------------------
print("Sentiment feature columns:")
for col in SENTIMENT_COLUMNS:
    print(f"   {col}")

print(f"\nTrain sentiment statistics:")
display(
    train_sentiment[SENTIMENT_COLUMNS].describe()
    .T.style.format("{:.4f}")
    .set_caption("Train Sentiment Feature Statistics")
)

Sentiment feature columns:
   doc_sentiment_score
   doc_sentiment_magnitude
   sentence_count_sentiment
   mean_sentence_score
   min_sentence_score
   max_sentence_score
   sentiment_variance
   entity_count
   entity_type_count

Train sentiment statistics:


,count,mean,std,min,25%,50%,75%,max
doc_sentiment_score,14442.0000,0.2810,0.2768,-0.9000,0.1000,0.3000,0.4000,0.9000
doc_sentiment_magnitude,14442.0000,2.1276,2.0369,0.0000,0.8000,1.6000,2.8000,32.0000
sentence_count_sentiment,14442.0000,5.1160,4.8127,1.0000,2.0000,4.0000,7.0000,84.0000
mean_sentence_score,14442.0000,0.2930,0.2740,-0.9000,0.1000,0.2750,0.4500,0.9000
min_sentence_score,14442.0000,-0.0875,0.4305,-0.9000,-0.4000,0.0000,0.1000,0.9000
max_sentence_score,14442.0000,0.6652,0.3224,-0.9000,0.5000,0.8000,0.9000,0.9000
sentiment_variance,14442.0000,0.1117,0.1027,0.0000,0.0100,0.1050,0.1650,0.8100
entity_count,14442.0000,11.1869,11.7408,0.0000,4.0000,8.0000,14.0000,207.0000
entity_type_count,14442.0000,2.9612,1.2899,0.0000,2.0000,3.0000,4.0000,7.0000


---
## 8. Assemble Text Feature Matrices

<div style="border-left: 4px solid #27ae60; padding: 8px 12px; margin: 8px 0; background: #2d4a37;">

Merge handcrafted text statistics, dense embeddings, and sentiment features into a single DataFrame per split, keyed on `PetID`.

</div>

In [15]:
# -- Build embedding DataFrames ------------------------------------------------
emb_col_names = [f"text_emb_{i}" for i in range(train_embeddings.shape[1])]

train_emb_df = pd.DataFrame(
    train_embeddings.astype(np.float32),
    columns=emb_col_names,
    index=train.index,
)
test_emb_df = pd.DataFrame(
    test_embeddings.astype(np.float32),
    columns=emb_col_names,
    index=test.index,
)

print(f"Embedding DataFrames:")
print(f"   Train: {train_emb_df.shape}")
print(f"   Test : {test_emb_df.shape}")

Embedding DataFrames:
   Train: (14993, 768)
   Test : (3972, 768)


In [16]:
# -- Assemble TRAIN feature matrix ---------------------------------------------
train_features = pd.DataFrame({"PetID": train["PetID"].values}, index=train.index)

# Add text statistics
for col in TEXT_STAT_COLUMNS:
    train_features[col] = train_text_stats[col].values

# Add embeddings
for col in emb_col_names:
    train_features[col] = train_emb_df[col].values

# Add sentiment features
train_features = train_features.merge(
    train_sentiment[["PetID"] + SENTIMENT_COLUMNS],
    on="PetID",
    how="left",
)
for col in SENTIMENT_COLUMNS:
    train_features[col] = train_features[col].fillna(0.0).astype(np.float32)

print(f"Train feature matrix: {train_features.shape}")
print(f"   Null values: {int(train_features.isna().sum().sum())}")

C:\Users\cliente\AppData\Local\Temp\ipykernel_10000\763864166.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_features[col] = train_emb_df[col].values
C:\Users\cliente\AppData\Local\Temp\ipykernel_10000\763864166.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_features[col] = train_emb_df[col].values
C:\Users\cliente\AppData\Local\Temp\ipykernel_10000\763864166.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Train feature matrix: (14993, 785)
   Null values: 0


C:\Users\cliente\AppData\Local\Temp\ipykernel_10000\763864166.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_features[col] = train_emb_df[col].values
C:\Users\cliente\AppData\Local\Temp\ipykernel_10000\763864166.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_features[col] = train_emb_df[col].values
C:\Users\cliente\AppData\Local\Temp\ipykernel_10000\763864166.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

In [17]:
# -- Assemble TEST feature matrix ----------------------------------------------
test_features = pd.DataFrame({"PetID": test["PetID"].values}, index=test.index)

# Add text statistics
for col in TEXT_STAT_COLUMNS:
    test_features[col] = test_text_stats[col].values

# Add embeddings
for col in emb_col_names:
    test_features[col] = test_emb_df[col].values

# Add sentiment features
test_features = test_features.merge(
    test_sentiment[["PetID"] + SENTIMENT_COLUMNS],
    on="PetID",
    how="left",
)
for col in SENTIMENT_COLUMNS:
    test_features[col] = test_features[col].fillna(0.0).astype(np.float32)

print(f"Test feature matrix: {test_features.shape}")
print(f"   Null values: {int(test_features.isna().sum().sum())}")

C:\Users\cliente\AppData\Local\Temp\ipykernel_10000\2714498046.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_features[col] = test_emb_df[col].values
C:\Users\cliente\AppData\Local\Temp\ipykernel_10000\2714498046.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_features[col] = test_emb_df[col].values
C:\Users\cliente\AppData\Local\Temp\ipykernel_10000\2714498046.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performan

Test feature matrix: (3972, 785)
   Null values: 0


C:\Users\cliente\AppData\Local\Temp\ipykernel_10000\2714498046.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_features[col] = test_emb_df[col].values
C:\Users\cliente\AppData\Local\Temp\ipykernel_10000\2714498046.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_features[col] = test_emb_df[col].values
C:\Users\cliente\AppData\Local\Temp\ipykernel_10000\2714498046.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performan

---
## 9. Feature Matrix Inspection

In [18]:
# -- Feature matrix overview ---------------------------------------------------
feature_cols_no_id = [c for c in train_features.columns if c != "PetID"]
n_text_stats = len(TEXT_STAT_COLUMNS)
n_emb_cols = len(emb_col_names)
n_sent_cols = len(SENTIMENT_COLUMNS)

print(f"Text feature matrix structure:")
print(f"   PetID             : 1 column (index key)")
print(f"   Text statistics   : {n_text_stats} columns")
print(f"   Embedding dims    : {n_emb_cols} columns (text_emb_0 .. text_emb_{n_emb_cols - 1})")
print(f"   Sentiment features: {n_sent_cols} columns")
print(f"   Total features    : {len(feature_cols_no_id)}")
print(f"\n   Train shape: {train_features.shape}")
print(f"   Test shape : {test_features.shape}")

Text feature matrix structure:
   PetID             : 1 column (index key)
   Text statistics   : 7 columns
   Embedding dims    : 768 columns (text_emb_0 .. text_emb_767)
   Sentiment features: 9 columns
   Total features    : 784

   Train shape: (14993, 785)
   Test shape : (3972, 785)


In [19]:
# -- Data types summary --------------------------------------------------------
print("Data type counts:")
print(train_features.dtypes.value_counts().to_string())
print(f"\nMemory usage:")
print(f"   Train: {train_features.memory_usage(deep=True).sum() / 1_048_576:.2f} MB")
print(f"   Test : {test_features.memory_usage(deep=True).sum() / 1_048_576:.2f} MB")

Data type counts:
float32    781
int32        3
str          1

Memory usage:
   Train: 45.08 MB
   Test : 11.94 MB


In [20]:
# -- Descriptive statistics for non-embedding features --------------------------
non_emb_cols = TEXT_STAT_COLUMNS + SENTIMENT_COLUMNS
display(
    train_features[non_emb_cols].describe()
    .T.style.format("{:.4f}")
    .set_caption("Train Text & Sentiment Features -- Descriptive Statistics")
)

,count,mean,std,min,25%,50%,75%,max
description_length,14993.0000,339.2214,373.3380,1.0000,117.0000,238.0000,431.0000,6664.0000
word_count,14993.0000,62.9522,69.2966,1.0000,21.0000,44.0000,81.0000,1257.0000
sentence_count,14993.0000,6.1625,5.1608,1.0000,3.0000,5.0000,8.0000,86.0000
avg_word_length,14993.0000,4.6152,1.9978,1.0000,4.1273,4.4127,4.8000,118.0000
uppercase_ratio,14993.0000,0.0397,0.0701,0.0000,0.0176,0.0258,0.0394,1.0000
punctuation_ratio,14993.0000,0.0397,0.0639,0.0000,0.0219,0.0302,0.0426,1.0000
digit_ratio,14993.0000,0.0043,0.0123,0.0000,0.0000,0.0000,0.0054,1.0000
doc_sentiment_score,14993.0000,0.2706,0.2767,-0.9000,0.1000,0.2000,0.4000,0.9000
doc_sentiment_magnitude,14993.0000,2.0494,2.0388,0.0000,0.8000,1.6000,2.8000,32.0000
sentence_count_sentiment,14993.0000,4.9280,4.8205,0.0000,2.0000,4.0000,6.0000,84.0000


In [21]:
# -- Embedding statistics (summary over dimensions) ----------------------------
train_emb_array = train_features[emb_col_names].values
print("Embedding dimension statistics (across all train samples):")
print(f"   Mean of means : {train_emb_array.mean():.6f}")
print(f"   Mean of stds  : {train_emb_array.std(axis=0).mean():.6f}")
print(f"   Min value     : {train_emb_array.min():.6f}")
print(f"   Max value     : {train_emb_array.max():.6f}")
print(f"   Norm range    : [{np.linalg.norm(train_emb_array, axis=1).min():.4f}, {np.linalg.norm(train_emb_array, axis=1).max():.4f}]")

Embedding dimension statistics (across all train samples):
   Mean of means : -0.000136
   Mean of stds  : 0.027326
   Min value     : -0.236361
   Max value     : 0.239533
   Norm range    : [1.0000, 1.0000]


In [22]:
# -- Train vs Test column alignment check --------------------------------------
train_cols = set(train_features.columns)
test_cols = set(test_features.columns)

only_train = train_cols - test_cols
only_test = test_cols - train_cols

print("Column alignment check:")
if not only_train and not only_test:
    print("   PASS: Train and test have identical column sets.")
else:
    if only_train:
        print(f"   Only in train: {only_train}")
    if only_test:
        print(f"   Only in test : {only_test}")

print(f"\nTrain: {train_features.shape} | Test: {test_features.shape}")

Column alignment check:
   PASS: Train and test have identical column sets.

Train: (14993, 785) | Test: (3972, 785)


---
## 10. Persist Feature Matrices & Schema

<div style="border-left: 4px solid #8e44ad; padding: 8px 12px; margin: 8px 0; background: #4a3554;">

Feature matrices are saved to `data/features/text/v1/` as versioned Parquet files. A `schema.json` records column metadata, model name, embedding dimensionality, row counts, and the config hash for reproducibility tracking.

</div>

In [23]:
# -- Save feature matrices to Parquet ------------------------------------------
train_path = save_parquet(train_features, FEATURES_DIR / "train.parquet")
test_path = save_parquet(test_features, FEATURES_DIR / "test.parquet")

print(f"Train features saved: {train_path}")
print(f"Test features saved : {test_path}")

10:08:00  INFO      Saved Parquet: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\text\v1\train.parquet (14993 rows, 61.22 MB)
10:08:00  INFO      Saved Parquet: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\text\v1\test.parquet (3972 rows, 15.82 MB)


Train features saved: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\text\v1\train.parquet
Test features saved : C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\text\v1\test.parquet


In [24]:
# -- Generate and save schema.json ---------------------------------------------
# Build descriptions for embedding columns
all_descriptions = dict(FEATURE_DESCRIPTIONS)
for col in emb_col_names:
    dim_idx = col.split("_")[-1]
    all_descriptions[col] = f"Sentence embedding dimension {dim_idx} (all-mpnet-base-v2)"

column_descriptors = build_column_descriptors(
    train_features[feature_cols_no_id],
    source="text",
    descriptions=all_descriptions,
)

config_for_hash = {
    "version": "v1",
    "modality": "text",
    "seed": SEED,
    "model_name": TEXT_CONFIG["model_name"],
    "embedding_dim": TEXT_CONFIG["embedding_dim"],
    "apply_pca": TEXT_CONFIG["apply_pca"],
    "feature_columns": feature_cols_no_id,
}

schema_metadata = {
    "version": "v1",
    "modality": "text",
    "model_name": TEXT_CONFIG["model_name"],
    "config_hash": compute_config_hash(config_for_hash),
    "n_rows_train": len(train_features),
    "n_rows_test": len(test_features),
    "n_features": len(feature_cols_no_id),
    "seed": SEED,
    "notes": (
        "Text features: 7 handcrafted statistics + 768 sentence embeddings "
        "(all-mpnet-base-v2) + 9 sentiment features from Google NLP API JSONs. "
        "No PCA applied."
    ),
}

schema_path = save_feature_schema(column_descriptors, schema_metadata, FEATURES_DIR / "schema.json")
print(f"Schema saved: {schema_path}")

10:08:00  INFO      Saved feature schema: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\text\v1\schema.json (784 columns, modality=text)


Schema saved: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\text\v1\schema.json


In [25]:
# -- Save extraction report ----------------------------------------------------
report = {
    "notebook": "08_feature_extraction_text",
    "model_name": TEXT_CONFIG["model_name"],
    "embedding_dim": TEXT_CONFIG["embedding_dim"],
    "apply_pca": TEXT_CONFIG["apply_pca"],
    "train_shape": list(train_features.shape),
    "test_shape": list(test_features.shape),
    "n_features": len(feature_cols_no_id),
    "n_text_stats": n_text_stats,
    "n_embedding_dims": n_emb_cols,
    "n_sentiment_features": n_sent_cols,
    "feature_columns": feature_cols_no_id,
    "preprocess_stats": {
        "train": train_preprocess_stats,
        "test": test_preprocess_stats,
    },
    "language_report": lang_report,
    "sentiment_coverage": {
        "train": round(float(train_coverage), 4),
        "test": round(float(test_coverage), 4),
    },
    "embedding_time_seconds": {
        "train": round(train_emb_time, 1),
        "test": round(test_emb_time, 1),
    },
    "config_hash": schema_metadata["config_hash"],
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)

print(f"Extraction report saved: {REPORT_PATH}")

Extraction report saved: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\reports\feature_extraction_text.json


---
## 11. Validation Gate -- G08

In [26]:
#  VALIDATION GATE -- G08
# ======================================================================

gate_results = []

# G08-1: Cleaned train/test loaded with Description and PetID columns
try:
    assert "PetID" in train.columns and "Description" in train.columns
    assert "PetID" in test.columns and "Description" in test.columns
    gate_results.append(("G08-1", "PASS", "Cleaned data loaded with Description and PetID columns"))
except AssertionError as e:
    gate_results.append(("G08-1", "FAIL", str(e)))

# G08-2: No NaN values in embedding columns
try:
    train_emb_nans = int(train_features[emb_col_names].isna().sum().sum())
    test_emb_nans = int(test_features[emb_col_names].isna().sum().sum())
    assert train_emb_nans == 0, f"Train has {train_emb_nans} NaN embedding values"
    assert test_emb_nans == 0, f"Test has {test_emb_nans} NaN embedding values"
    gate_results.append(("G08-2", "PASS", "No NaN values in embedding columns"))
except AssertionError as e:
    gate_results.append(("G08-2", "FAIL", str(e)))

# G08-3: Embedding dimension matches model specification
try:
    actual_emb_dim = len(emb_col_names)
    assert actual_emb_dim == TEXT_CONFIG["embedding_dim"], (
        f"Embedding dim {actual_emb_dim} != {TEXT_CONFIG['embedding_dim']}"
    )
    gate_results.append(("G08-3", "PASS", f"Embedding dimension matches: {actual_emb_dim}"))
except AssertionError as e:
    gate_results.append(("G08-3", "FAIL", str(e)))

# G08-4: Train and test feature matrices have identical column sets
try:
    assert set(train_features.columns) == set(test_features.columns), (
        f"Column mismatch: train_only={train_cols - test_cols}, test_only={test_cols - train_cols}"
    )
    gate_results.append(("G08-4", "PASS", "Train and test have identical column sets"))
except AssertionError as e:
    gate_results.append(("G08-4", "FAIL", str(e)))

# G08-5: Row counts preserved
try:
    assert len(train_features) == EXPECTED_TRAIN_ROWS, (
        f"Train rows: {len(train_features)} != {EXPECTED_TRAIN_ROWS}"
    )
    assert len(test_features) == EXPECTED_TEST_ROWS, (
        f"Test rows: {len(test_features)} != {EXPECTED_TEST_ROWS}"
    )
    gate_results.append(("G08-5", "PASS", f"Row counts preserved: {EXPECTED_TRAIN_ROWS:,} / {EXPECTED_TEST_ROWS:,}"))
except AssertionError as e:
    gate_results.append(("G08-5", "FAIL", str(e)))

# G08-6: PetID uniqueness maintained
try:
    assert train_features["PetID"].is_unique, "Train PetID not unique"
    assert test_features["PetID"].is_unique, "Test PetID not unique"
    gate_results.append(("G08-6", "PASS", "PetID uniqueness maintained"))
except AssertionError as e:
    gate_results.append(("G08-6", "FAIL", str(e)))

# G08-7: Sentiment features present for >= 95% of PetIDs
try:
    sent_coverage_train = float(train_sentiment["PetID"].isin(train["PetID"]).sum() / len(train))
    sent_coverage_test = float(test_sentiment["PetID"].isin(test["PetID"]).sum() / len(test))
    if sent_coverage_train >= 0.95 and sent_coverage_test >= 0.95:
        gate_results.append(("G08-7", "PASS", f"Sentiment coverage: train={sent_coverage_train:.2%}, test={sent_coverage_test:.2%}"))
    else:
        gate_results.append(("G08-7", "WARN", f"Sentiment coverage below 95%: train={sent_coverage_train:.2%}, test={sent_coverage_test:.2%}"))
except Exception as e:
    gate_results.append(("G08-7", "WARN", str(e)))

# G08-8: No target leakage
try:
    assert "AdoptionSpeed" not in train_features.columns, "AdoptionSpeed in train features"
    assert "AdoptionSpeed" not in test_features.columns, "AdoptionSpeed in test features"
    gate_results.append(("G08-8", "PASS", "No target leakage in feature matrices"))
except AssertionError as e:
    gate_results.append(("G08-8", "FAIL", str(e)))

# G08-9: Parquet files exist and are loadable
try:
    reload_train = pd.read_parquet(FEATURES_DIR / "train.parquet")
    reload_test = pd.read_parquet(FEATURES_DIR / "test.parquet")
    gate_results.append(("G08-9", "PASS", "Parquet files exist and are loadable"))
except Exception as e:
    gate_results.append(("G08-9", "FAIL", str(e)))

# G08-10: schema.json exists, valid JSON, records model name
try:
    with open(FEATURES_DIR / "schema.json", "r", encoding="utf-8") as f:
        loaded_schema = json.load(f)
    assert "columns" in loaded_schema, "schema.json missing 'columns' key"
    assert "version" in loaded_schema, "schema.json missing 'version' key"
    assert loaded_schema.get("model_name") == TEXT_CONFIG["model_name"], (
        f"Model name mismatch in schema: {loaded_schema.get('model_name')}"
    )
    gate_results.append(("G08-10", "PASS", "schema.json exists, valid JSON, records model name"))
except Exception as e:
    gate_results.append(("G08-10", "FAIL", str(e)))

# G08-11: No all-zero embedding vectors (degenerate check)
try:
    train_zero = int(np.all(train_features[emb_col_names].values == 0, axis=1).sum())
    test_zero = int(np.all(test_features[emb_col_names].values == 0, axis=1).sum())
    if train_zero == 0 and test_zero == 0:
        gate_results.append(("G08-11", "PASS", "No degenerate (all-zero) embedding vectors"))
    else:
        gate_results.append(("G08-11", "WARN", f"Degenerate embeddings: train={train_zero}, test={test_zero}"))
except Exception as e:
    gate_results.append(("G08-11", "WARN", str(e)))

# G08-12: Feature count matches schema n_features
try:
    schema_n = loaded_schema["n_features"]
    actual_n = len(feature_cols_no_id)
    assert schema_n == actual_n, f"Schema says {schema_n}, actual is {actual_n}"
    gate_results.append(("G08-12", "PASS", f"Feature count matches schema: {actual_n}"))
except AssertionError as e:
    gate_results.append(("G08-12", "WARN", str(e)))

# -- Parquet round-trip integrity (bonus, aligned with G07 pattern) ----------
try:
    assert reload_train.shape == train_features.shape, (
        f"Shape mismatch: {reload_train.shape} != {train_features.shape}"
    )
    assert reload_test.shape == test_features.shape, (
        f"Shape mismatch: {reload_test.shape} != {test_features.shape}"
    )
    assert list(reload_train.columns) == list(train_features.columns), "Column order mismatch"
    gate_results.append(("G08-RT", "PASS", "Parquet round-trip integrity verified"))
except AssertionError as e:
    gate_results.append(("G08-RT", "FAIL", str(e)))

# -- Print gate summary -------------------------------------------------------
print("=" * 70)
print("  VALIDATION GATE -- G08 RESULTS")
print("=" * 70)
for gid, status, msg in gate_results:
    if status == "PASS":
        icon = "[PASS]"
    elif "WARN" in status:
        icon = "[WARN]"
    else:
        icon = "[FAIL]"
    print(f"  {icon} {gid}: {msg}")

passed = sum(1 for _, s, _ in gate_results if s == "PASS")
warned = sum(1 for _, s, _ in gate_results if "WARN" in s)
failed = sum(1 for _, s, _ in gate_results if "FAIL" in s)
total = len(gate_results)
print(f"\n  Result: {passed}/{total} passed, {warned} warnings, {failed} failures.")
print("=" * 70)

# Halt on critical failures
critical_ids = {"G08-1", "G08-2", "G08-3", "G08-4", "G08-5", "G08-6", "G08-8", "G08-9", "G08-10", "G08-RT"}
critical_failures = [g for g in gate_results if g[1] == "FAIL" and g[0] in critical_ids]
assert len(critical_failures) == 0, f"Critical gate failures: {critical_failures}"

  VALIDATION GATE -- G08 RESULTS
  [PASS] G08-1: Cleaned data loaded with Description and PetID columns
  [PASS] G08-2: No NaN values in embedding columns
  [PASS] G08-3: Embedding dimension matches: 768
  [PASS] G08-4: Train and test have identical column sets
  [PASS] G08-5: Row counts preserved: 14,993 / 3,972
  [PASS] G08-6: PetID uniqueness maintained
  [PASS] G08-7: Sentiment coverage: train=96.32%, test=97.31%
  [PASS] G08-8: No target leakage in feature matrices
  [PASS] G08-9: Parquet files exist and are loadable
  [PASS] G08-10: schema.json exists, valid JSON, records model name
  [PASS] G08-11: No degenerate (all-zero) embedding vectors
  [PASS] G08-12: Feature count matches schema: 784
  [PASS] G08-RT: Parquet round-trip integrity verified

  Result: 13/13 passed, 0 warnings, 0 failures.


---
## 12. Feature Summary

In [27]:
# -- Feature catalog with descriptions -----------------------------------------
catalog_rows = []

# Text statistics
for col in TEXT_STAT_COLUMNS:
    catalog_rows.append({
        "Feature": col,
        "Group": "text_stat",
        "Dtype": str(train_features[col].dtype),
        "Nulls": int(train_features[col].isna().sum()),
        "Description": all_descriptions.get(col, ""),
    })

# Embeddings (summary row)
catalog_rows.append({
    "Feature": f"text_emb_0 .. text_emb_{n_emb_cols - 1}",
    "Group": "embedding",
    "Dtype": "float32",
    "Nulls": int(train_features[emb_col_names].isna().sum().sum()),
    "Description": f"{n_emb_cols}-dim sentence embeddings (all-mpnet-base-v2)",
})

# Sentiment features
for col in SENTIMENT_COLUMNS:
    catalog_rows.append({
        "Feature": col,
        "Group": "sentiment",
        "Dtype": str(train_features[col].dtype),
        "Nulls": int(train_features[col].isna().sum()),
        "Description": all_descriptions.get(col, ""),
    })

catalog_df = pd.DataFrame(catalog_rows)
display(
    catalog_df.style
    .set_caption("Text Feature Catalog (v1)")
    .set_properties(**{"text-align": "left"})
    .hide(axis="index")
)

Feature,Group,Dtype,Nulls,Description
description_length,text_stat,int32,0,Character length of preprocessed description
word_count,text_stat,int32,0,Word count of preprocessed description
sentence_count,text_stat,int32,0,Approximate sentence count (split on .!?)
avg_word_length,text_stat,float32,0,Mean word length in characters
uppercase_ratio,text_stat,float32,0,Fraction of uppercase characters in description
punctuation_ratio,text_stat,float32,0,Fraction of punctuation characters in description
digit_ratio,text_stat,float32,0,Fraction of digit characters in description
text_emb_0 .. text_emb_767,embedding,float32,0,768-dim sentence embeddings (all-mpnet-base-v2)
doc_sentiment_score,sentiment,float32,0,Document-level sentiment polarity (-1.0 to +1.0)
doc_sentiment_magnitude,sentiment,float32,0,Document-level sentiment intensity


---
## Summary

<div style="border-left: 4px solid #27ae60; padding: 10px 15px; margin: 10px 0; background: #232a3b;">

**Text Feature Extraction -- Complete**

1. **Text preprocessing** -- Null/empty handling, HTML/URL removal, whitespace normalization, NFKD Unicode normalization. Casing preserved for transformer compatibility.
2. **Language detection** -- Sampled 1,000 descriptions for language distribution assessment.
3. **Handcrafted text statistics** -- 7 features: `description_length`, `word_count`, `sentence_count`, `avg_word_length`, `uppercase_ratio`, `punctuation_ratio`, `digit_ratio`.
4. **Sentence embeddings** -- 768-dimensional dense embeddings via `all-mpnet-base-v2` (highest quality sentence-transformer). No PCA applied.
5. **Sentiment features** -- 9 features aggregated from Google NLP API JSONs: document scores, sentence-level statistics (mean, min, max, variance), entity counts.

**Artifacts produced:**
- `data/features/text/v1/train.parquet` (14,993 rows)
- `data/features/text/v1/test.parquet` (3,972 rows)
- `data/features/text/v1/schema.json`
- `reports/feature_extraction_text.json`

</div>

**Next:** Notebook `09_feature_extraction_images` (image features) or `10_feature_integration` (cross-modality feature fusion).